# 9.3 Stage 3: Extend Wrist (Expanded)

Simulates hand opening by extending wrist joints.

---

```python id="0kx9rc"
elif t < 3*d:
    self.log_stage(3, "Extending wrist (simulated hand open)")

    r = (t - 2*d) / d
    for i, j in enumerate(self.joints):
        self.low_cmd.motor_cmd[j].q = self.interp(self.target_raise[i], self.target_extend[i], r)
        self.low_cmd.motor_cmd[j].dq = 0
        self.low_cmd.motor_cmd[j].kp = self.kp
        self.low_cmd.motor_cmd[j].kd = self.kd
        self.low_cmd.motor_cmd[j].tau = 0
```

---

## 🧠 Big Picture: What Is This Stage Doing?

This stage tells the robot:

```text id="9y2nrf"
"Starting from the raised arm pose, adjust the wrist to simulate opening the hand"
```

---

### Key difference from Stage 2:

| Stage   | Start        | End           |
| ------- | ------------ | ------------- |
| Stage 2 | initial_pose | target_raise  |
| Stage 3 | target_raise | target_extend |

---

> 🔥 You are now **chaining trajectories**

---

## ⏱️ When Does This Stage Run?

```python
elif t < 3*d:
```

---

### Time window:

```text
2d → 3d
```

If `d = 3s`:

```text
6s → 9s
```

---

## 🔄 Step 1: Compute Interpolation Ratio

```python
r = (t - 2*d) / d
```

---

### Behavior:

| Time     | r   |
| -------- | --- |
| t = 2d   | 0   |
| t = 2.5d | 0.5 |
| t = 3d   | 1   |

---

### 🧠 Interpretation

```text
r = progress through wrist extension motion
```

---

## 🎯 Step 2: Change the Start and End States

```python
self.interp(self.target_raise[i], self.target_extend[i], r)
```

---

### This is the most important line in this stage

Instead of:

```python
interp(initial_pose, target_raise)
```

you now have:

```python
interp(target_raise, target_extend)
```

---

### Meaning:

```text
Continue motion from previous goal → not from original pose
```

---

## 🧠 Why This Matters

This creates:

> A **continuous multi-stage trajectory**

---

Without this chaining:

* Motion would reset
* Jumps would occur
* Behavior would look unnatural

---

## 🦾 What Is Actually Moving?

### Compare target vectors:

```python
target_raise  = [-0.3, 0.2, 0.0, -1.0, 0.0, 0.5, 0.0]
target_extend = [-0.3, 0.2, 0.0, -1.0, 0.0, 1.0, 0.2]
```

---

### Differences:

| Joint      | Change    |
| ---------- | --------- |
| WristPitch | 0.5 → 1.0 |
| WristYaw   | 0.0 → 0.2 |

---

### Interpretation:

```text
Only wrist joints are significantly moving
```

---

### Result:

* Arm stays raised
* Wrist extends and rotates
* Looks like “hand opening”

---

## ⚙️ PD Control (Same as Before)

```python
dq = 0
kp = self.kp
kd = self.kd
tau = 0
```

---

### Control law:

[
\tau = K_p (q_{target} - q) + K_d (0 - \dot{q})
]

---

### Behavior:

* Position error → drives motion
* Damping → smooths motion

---

## 🔄 Dynamic Behavior Over Time

Each loop:

```text
r increases
→ q_target shifts slightly
→ torque generated
→ wrist moves gradually
```

---

### Result:

```text
Smooth wrist extension
```

---

## 🧠 Control Architecture Insight

You are implementing:

> **Piecewise trajectory generation**

---

### Each stage:

```text
Defines a segment of motion
```

---

### Combined:

```text
Full behavior = sequence of segments
```

---

## ⚠️ Important Subtlety

Even though only wrist joints change:

```text
ALL joints are still controlled
```

---

### Why?

Because you still send:

```python
for all joints:
    cmd.q = ...
```

---

### This ensures:

* Shoulder stays stable
* Elbow stays stable
* Only wrist moves

---

## 🧠 System Stability Insight

If you did NOT command all joints:

```text
Uncontrolled joints could drift
```

---

## 🤖 RL Interpretation

This stage represents:

```text
A higher-level behavior composed of smaller actions
```

---

### In RL terms:

* Stage 2 = reach position
* Stage 3 = manipulate object (simulated)

---

### You are approximating:

```text
Task decomposition
```

---

## 🔬 Engineering Insight

This stage shows:

> You don’t need full motion planning to get meaningful behavior

---

With just:

* interpolation
* PD control
* staged targets

You get:

* coordinated motion
* task-like behavior

---

## 🔄 Analogy

Think of this like:

```text
You raise your arm → then open your hand
```

---

You don’t:

* restart from neutral
* you continue from the current pose

---

## 🔥 Hidden Design Pattern

This stage demonstrates:

> **Sequential motion composition**

---

Which is foundational for:

* robotics
* animation
* RL policies
* behavior trees

---

## 🚀 Summary

This stage:

| Step                | Role                     |
| ------------------- | ------------------------ |
| Compute `r`         | Progress through stage   |
| Interpolate         | Between two target poses |
| Maintain PD control | Stable motion            |
| Chain trajectory    | Continue previous motion |

---

### Result:

```text
Arm remains raised while wrist extends smoothly
```

---

> 🔥 This stage introduces the concept of **multi-stage behaviors**, which is essential for both robotics and reinforcement learning.

